In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet('../Data/processed/features_final.parquet')
print(df.shape)

In [ ]:
# Columnas categoricas
cat_cols = df.select_dtypes(include='str').columns.tolist()
print(f'Total de columnas categoricas: {len(cat_cols)}')
print(cat_cols)

In [ ]:
for col in cat_cols:
    print(f'{col}: {df[col].nunique()} categorías')

Para las columnas con alta cardinalidad usaremos label encoding, que asigna un entero a cada categoría. Para el resto usaremos one-hot encoding que crea una columna binaria por cada categoria.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']:
    df[col] = le.fit_transform(df[col].astype('str'))

In [ ]:
# Verificamos que la transformacion haya sido correcta
print(df[['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']].dtypes)

In [ ]:
cat_cols_ohe = [c for c in df.select_dtypes(include='str').columns]

df = pd.get_dummies(df, columns=cat_cols_ohe, drop_first=True)

df.head()

In [ ]:
# Separacion de datos en variables y target
X = df.copy().drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = df['TARGET'].copy()

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Escalamos los datos
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_prob_dummy = dummy.predict_proba(X_test)[:,1]

auc_dummy = roc_auc_score(y_test, y_prob_dummy)
print(f'Baseline AUC-ROC: {auc_dummy:.4f}')

Primer modelo real: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr = LogisticRegression(random_state=42, max_iter=1000)

scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='roc_auc')

print(f'AUC-ROC por fold: {scores.round(4)}')
print(f'AUC-ROC medio: {scores.mean():.4f} (+/- {scores.std():.4f})')

Segundo modelo: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')

scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring='roc_auc')

print(f'AUC-ROC por fold: {scores_rf.round(4)}')
print(f'AUC-ROC medio: {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})')

Tercer modelo: LightGBM

In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    verbose=-1
)

scores_lgb = cross_val_score(lgbm, X_train, y_train, cv=5, scoring='roc_auc')

print(f'AUC-ROC por fold: {scores_lgb.round(4)}')
print(f'AUC-ROC medio: {scores_lgb.mean():.4f} (+/- {scores_lgb.std():.4f})')

In [ ]:
lgbm_final = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    verbose=-1
)

lgbm_final.fit(X_train, y_train)
y_prob = lgbm_final.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(recall, precision, label=f'LightGBM (AP = {ap:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall')
ax.legend()
plt.tight_layout()
plt.savefig('../Reports/figures/09_precision_recall.png')
plt.show()

Mejor umbral para Precision y Recall

In [ ]:
from sklearn.metrics import f1_score

'''
Para reducir el coste computacional al calcular los f1 para cada umbral, se crea un conjunto de 200 umbrales, con el que se hará la comparación convirtiendo las probabilidades y los umbrales en matrices. Luego calculamos el F1 para cada columna de esta matriz resultante.
'''

thresholds_reducidos = np.linspace(0.01, 0.99, 200)

predicciones = (y_prob[:, np.newaxis] >= thresholds_reducidos[np.newaxis, :]).astype(int) 

f1_scores = [f1_score(y_test, predicciones[:, i]) for i in range(predicciones.shape[1])]

# for threshold in thresholds:
#     y_prob_thresh = (y_prob >= threshold).astype(int)
#     f1_scores.append(f1_score(y_test, y_prob_thresh))
    
umbral_optimo = thresholds_reducidos[np.argmax(f1_scores)]
f1_optimo = max(f1_scores)

print(f'Umbral optimo: {umbral_optimo:.4f}')
print(f'f1-score optimo: {f1_optimo:.4f}')

In [ ]:
y_pred_dummy_class = dummy.predict(X_test)
f1_dummy = f1_score(y_test, y_pred_dummy_class)
print(f"F1 Baseline: {f1_dummy:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_final = (y_prob >= umbral_optimo).astype(int)

print(classification_report(y_test, y_pred_final, target_names=['Paga', 'Impago']))

cm = confusion_matrix(y_test, y_pred_final)
print(cm)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))

ax.plot(thresholds, precision[:-1], label='Precision')
ax.plot(thresholds, recall[:-1], label='Recall')
ax.plot(thresholds, f1_scores, label='F1')

ax.axvline(x=umbral_optimo, color='red', linestyle='--', label=f'Umbral optimo ({umbral_optimo:.4f})')

ax.set_title('Precision, Recall y F1 por umbral')
ax.set_xlabel('Umbral')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.savefig('../Reports/figures/10_threshold_tuning.png', dpi=150)
plt.show()

In [ ]:
y_pred_060 = (y_prob >= 0.50).astype(int)
print(classification_report(y_test, y_pred_060, target_names=['Paga', 'Impago']))

Con un umbral de decision de 0.50 conseguimos aumentar el recall a 0.65.

#### Explicabilidad con SHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(lgbm_final)

shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=X.columns.tolist(), max_display=15)

In [ ]:
features_importance = pd.DataFrame({
    'feature': X.columns.tolist(),
    'shap_mean': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_mean', ascending=False).head(10)

features_importance

Las fuestes externas de scoring son el predictor más importante para el modelo, por encima de variables internas como tipo de empleo o ingresos.

In [ ]:
# Caso de impago predicho correctamente
impagos_correctos = np.where((y_pred_final == 1) & (y_test.values == 1))[0]

caso = impagos_correctos[0]

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[caso],
        base_values= explainer.expected_value,
        data=X_test[caso],
        feature_names= X.columns.tolist()
    )
)

Resumen de modelos

In [ ]:
resumen_modelos = pd.DataFrame({
    'Modelo': ['DummyClassifier', 'Random Forest', 'Logistic Regression', 'LightGBM'],
    'AUC-ROC': [0.5, scores_rf.mean(), scores.mean(), scores_lgb.mean()],
    'Std': [0, scores_rf.std(), scores.std(), scores_lgb.std()]
})

resumen_modelos.sort_values('AUC-ROC', ascending=False)

Despues de haber entrenado y probado los tres modelos, hemos obtenido los siguientes resultados:
- LightGBM fue el mejor modelo con mejores resultados con un AUC-ROC = 0.7589.
- Se pudo determinar que con umbral de decisión de 0.50, basado en la curva Precision-Recall, se logra aumentar el Recall para detectar el 65% de los impagos reales.
- Las variables de mayor peso para el modelo LightGBM fueron los scoring de credito externos para predecir si un cliente pagará o no, con un shap_medio = 0.283.